# Import Packages

In [95]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [96]:
import sys
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon
import folium
import json
import time
import numpy as np
import h3
from folium.plugins import HeatMap
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
import time

In [97]:
sys.path.append('src')

In [125]:
from fetch_data import *
from data_io import *
from data_prep import *
from scoring import *
from threshold_clustering import *
from dbscan_clustering import *
from visualization import *

# User Inputs

In [99]:
# need to update it to circular boundaries based on user defined center and radius

#define boundaries
ATLANTA_BBOX  = [33.64, -84.55, 33.89, -84.29]

#need to make it scalable for more features later on

user_weights = {
    'restaurant': 0.5,
    'park': 0.1,
    'clinic': 0.4
}

#define radius of the search area in km, optional - can also define center
user_radius_km = 12

# Query Data

In [6]:
all_pois = []
all_pois = query_restaurant_data(ATLANTA_BBOX, all_pois)
all_pois = query_park_data(ATLANTA_BBOX, all_pois)
all_pois = query_hospital_and_clinic_data(ATLANTA_BBOX, all_pois)
print(f"Total POIs fetched: {len(all_pois)}")

Fetching restaurant data


KeyboardInterrupt: 

# Query Save & Load

In [7]:
name_of_the_file = "atlanta_pois"

In [14]:
save_pois(all_pois, name_of_the_file)

KeyError: 'lon'

In [100]:
df_pois = load_pois(name_of_the_file)

Loaded 1784
Summary by type:
type
restaurant    1029
park           504
cafe           185
clinic          48
hospital        18
Name: count, dtype: int64


# Data Prep - Data Points

In [126]:
# either use boundaries or radius function to create hex grids
# hexagons = create_hex_grids_with_boundaries(df_pois)
hexagons = create_hex_grids_with_radius(df_pois, radius_km=user_radius_km, size_of_grid = 8)



Using circular boundary: center (33.7490, -84.3880), radius 12 km
Generated 890 hexagons (before filtering)
Filtered to 536 hexagons within 12 km of center


# Data Prep - Features

In [128]:
df_hexagons = calculate_accessibility_scores(hexagons, df_pois)

Calculating accessibility scores for each hexagon...
  Processing hexagon 0/536...
  Processing hexagon 50/536...
  Processing hexagon 100/536...
  Processing hexagon 150/536...
  Processing hexagon 200/536...
  Processing hexagon 250/536...
  Processing hexagon 300/536...
  Processing hexagon 350/536...
  Processing hexagon 400/536...
  Processing hexagon 450/536...
  Processing hexagon 500/536...

✓ Calculated accessibility scores for 536 hexagons

Accessibility Score Statistics:
       restaurant_accessibility  park_accessibility  clinic_accessibility
count                536.000000          536.000000            536.000000
mean                   5.644569            5.242794              1.532971
std                    9.308070            3.989966              1.576083
min                    0.000000            0.154927              0.000000
25%                    0.124571            2.318832              0.418872
50%                    1.717854            4.127623              0.85

In [129]:
# df_hexagons = apply_user_weights(df_hexagons, user_weights)

df_hexagons = apply_user_weights(df_hexagons, user_weights, smooth_before_weighting=True, neighbor_weight=0.3)



print(df_hexagons.head())


APPLYING USER WEIGHTS
User preferences: {'restaurant': 0.5, 'park': 0.1, 'clinic': 0.4}
Sum of weights: 1.00 (should be 1.0)

Applying spatial smoothing to 3 score columns...
Neighbor weight: 0.30
  Smoothing hexagon 0/536...
  Smoothing hexagon 50/536...
  Smoothing hexagon 100/536...
  Smoothing hexagon 150/536...
  Smoothing hexagon 200/536...
  Smoothing hexagon 250/536...
  Smoothing hexagon 300/536...
  Smoothing hexagon 350/536...
  Smoothing hexagon 400/536...
  Smoothing hexagon 450/536...
  Smoothing hexagon 500/536...
✓ Spatial smoothing complete

User Match Score Statistics:
count    536.000000
mean       0.162353
std        0.178165
min        0.000891
25%        0.044712
50%        0.087538
75%        0.242190
max        0.953797
Name: user_match_score, dtype: float64
            hex_id        lat        lon  restaurant_accessibility  \
0  8844c13205fffff  33.832688 -84.354754                  6.472448   
1  8844c1aa61fffff  33.669164 -84.387398                  0.183164

# Experiment 1: Threshold Clustering

In [130]:
df_threshold = cluster_based_on_score(df_hexagons)
df_threshold.head()


Thresholds: High = 0.184, Medium = 0.055

Suitability Distribution:
suitability_label
Okay             182
Less Suitable    177
Most Suitable    177
Name: count, dtype: int64

SUITABILITY TIER CHARACTERISTICS

Most Suitable (177 hexagons):
  Match Score Range: 0.185 - 0.954
  Avg restaurant Access: 14.297
  Avg park Access: 9.418
  Avg clinic Access: 3.397

Okay (182 hexagons):
  Match Score Range: 0.055 - 0.184
  Avg restaurant Access: 2.461
  Avg park Access: 4.118
  Avg clinic Access: 0.961

Less Suitable (177 hexagons):
  Match Score Range: 0.001 - 0.055
  Avg restaurant Access: 0.291
  Avg park Access: 2.244
  Avg clinic Access: 0.261


,hex_id,lat,lon,restaurant_accessibility,park_accessibility,clinic_accessibility,restaurant_norm,park_norm,clinic_norm,user_match_score,suitability,suitability_label
0,8844c13205fffff,33.832688,-84.354754,6.472448,5.931081,1.351004,0.113310,0.290899,0.192102,0.162586,1,Okay
1,8844c1aa61fffff,33.669164,-84.387398,0.183164,2.546198,0.331047,0.003207,0.118221,0.047072,0.032254,2,Less Suitable
2,8844c1a837fffff,33.760744,-84.358751,32.377703,15.217928,4.039757,0.566821,0.764664,0.574422,0.589645,0,Most Suitable
3,8844c1a131fffff,33.711909,-84.465248,0.198940,1.921649,0.341870,0.003483,0.086360,0.048611,0.029822,2,Less Suitable
4,8844c1a933fffff,33.755925,-84.302435,5.618555,8.282978,1.860461,0.098361,0.410880,0.264543,0.196086,0,Most Suitable


In [131]:
threshold_map_name = "data/output_data/atlanta_threshold_map.html"
map_threshold = create_suitability_map(df_threshold, user_weights)
map_threshold.save(threshold_map_name)

Adding hexagons to map...
  Added 0/536 hexagons...
  Added 50/536 hexagons...
  Added 100/536 hexagons...
  Added 150/536 hexagons...
  Added 200/536 hexagons...
  Added 250/536 hexagons...
  Added 300/536 hexagons...
  Added 350/536 hexagons...
  Added 400/536 hexagons...
  Added 450/536 hexagons...
  Added 500/536 hexagons...


# Experiment 2: DBSCAN Clustering

In [57]:
df_dbscan = dbscan_score_clustering(df_hexagons, eps=0.1, min_samples=3)


DBSCAN CLUSTERING (Score-Based)
Parameters: eps=0.1, min_samples=3

Results:
  Clusters found: 6
  Noise points: 6
  Noise/Uncertain: 6 hexagons, avg score = 0.873
  Cluster 0: 500 hexagons, avg score = 0.126
  Cluster 1: 6 hexagons, avg score = 0.585
  Cluster 2: 4 hexagons, avg score = 0.694
  Cluster 3: 7 hexagons, avg score = 0.631
  Cluster 4: 8 hexagons, avg score = 0.519
  Cluster 5: 5 hexagons, avg score = 0.788


In [58]:
cluster_colors = get_cluster_colors(df_dbscan)

In [59]:
dbscan_map_name = "data/output_data/atlanta_dbscan_map.html"
map_dbscan = create_dbscan_map(df_dbscan, user_weights, cluster_colors=cluster_colors, use_heatmap=True, heatmap_radius=15)
map_dbscan.save(dbscan_map_name)

Adding hexagons to map...
  Added 0/536 hexagons...
  Added 50/536 hexagons...
  Added 100/536 hexagons...
  Added 150/536 hexagons...
  Added 200/536 hexagons...
  Added 250/536 hexagons...
  Added 300/536 hexagons...
  Added 350/536 hexagons...
  Added 400/536 hexagons...
  Added 450/536 hexagons...
  Added 500/536 hexagons...
Adding heatmap overlay for smooth visualization...


# Experiment 3 - Sptial DBSCAN

In [92]:
df_dbscan_spatial = dbscan_spatial_clustering(
    df_hexagons,
    eps=0.2,
    min_samples=3,
    spatial_weight=0.4
)


DBSCAN CLUSTERING (Spatially-Aware)
Parameters: eps=0.2, min_samples=3, spatial_weight=0.4

Results:
  Clusters found: 4
  Noise points: 5
  Noise/Uncertain: 5 hexagons
  Region 0: 520 hexagons, avg score = 0.146, extent = 33.1 km
  Region 1: 5 hexagons, avg score = 0.788, extent = 3.7 km
  Region 2: 3 hexagons, avg score = 0.930, extent = 1.7 km
  Region 3: 3 hexagons, avg score = 0.517, extent = 3.4 km


In [69]:
cluster_colors_spatial = get_cluster_colors(df_dbscan_spatial)

In [70]:
map_dbscan_spatial = create_dbscan_map(
    df_dbscan_spatial,
    user_weights,
    cluster_colors=cluster_colors_spatial,
    use_heatmap=True,
    heatmap_radius=15
)

spatial_dbscan_map_name = "data/output_data/atlanta_spatial_dbscan_map.html"
map_dbscan_spatial.save(spatial_dbscan_map_name)

Adding hexagons to map...
  Added 0/536 hexagons...
  Added 50/536 hexagons...
  Added 100/536 hexagons...
  Added 150/536 hexagons...
  Added 200/536 hexagons...
  Added 250/536 hexagons...
  Added 300/536 hexagons...
  Added 350/536 hexagons...
  Added 400/536 hexagons...
  Added 450/536 hexagons...
  Added 500/536 hexagons...
Adding heatmap overlay for smooth visualization...
